# Simple Harmonic Oscillator (Complete Project)

This notebook numerically solves and visualizes a spring-mass system with optional damping.

We model:

$$m\ddot{x} + c\dot{x} + kx = 0$$

Features:
- Numerical integration (Euler-Cromer)
- Displacement, velocity, phase portrait, and energy plots
- Damping regime detection
- Interactive sliders for real-time exploration

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Checkbox, VBox, HBox, Output
from IPython.display import display, Markdown

print('Environment ready for SHO simulation.')

Environment ready for SHO simulation.


In [2]:
def simulate_sho(m, k, c, x0, v0, t_max, dt):
    n = int(t_max / dt) + 1
    t = np.linspace(0, t_max, n)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    v[0] = v0

    # Euler-Cromer integration (stable for oscillators)
    for i in range(n - 1):
        a = -(c / m) * v[i] - (k / m) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i + 1] * dt

    kinetic = 0.5 * m * v**2
    potential = 0.5 * k * x**2
    total = kinetic + potential

    return t, x, v, kinetic, potential, total


def damping_regime(m, k, c):
    c_critical = 2 * np.sqrt(k * m)
    if np.isclose(c, c_critical, rtol=1e-3, atol=1e-6):
        regime = 'Critically damped'
    elif c < c_critical:
        regime = 'Underdamped'
    else:
        regime = 'Overdamped'
    return regime, c_critical


# Controls
mass_slider = FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Mass m')
spring_slider = FloatSlider(value=4.0, min=0.1, max=30.0, step=0.1, description='Spring k')
damp_slider = FloatSlider(value=0.2, min=0.0, max=8.0, step=0.05, description='Damping c')
x0_slider = FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='x(0)')
v0_slider = FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description='v(0)')
tmax_slider = FloatSlider(value=20.0, min=2.0, max=100.0, step=1.0, description='t max')
dt_slider = FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001, readout_format='.3f', description='dt')
show_components = Checkbox(value=True, description='Show K & U')

ui = VBox([
    mass_slider,
    spring_slider,
    damp_slider,
    HBox([x0_slider, v0_slider]),
    HBox([tmax_slider, dt_slider]),
    show_components,
])
display(ui)

out = Output()


def update_plot(*_):
    with out:
        out.clear_output(wait=True)

        m = mass_slider.value
        k = spring_slider.value
        c = damp_slider.value
        x0 = x0_slider.value
        v0 = v0_slider.value
        t_max = tmax_slider.value
        dt = dt_slider.value

        t, x, v, ke, pe, te = simulate_sho(m, k, c, x0, v0, t_max, dt)
        regime, c_critical = damping_regime(m, k, c)
        omega0 = np.sqrt(k / m)

        fig, axs = plt.subplots(2, 2, figsize=(12, 7))

        axs[0, 0].plot(t, x, color='tab:blue', lw=1.8)
        axs[0, 0].set_title('Displacement x(t)')
        axs[0, 0].set_xlabel('Time')
        axs[0, 0].set_ylabel('x')
        axs[0, 0].grid(True, alpha=0.3)

        axs[0, 1].plot(t, v, color='tab:orange', lw=1.8)
        axs[0, 1].set_title('Velocity v(t)')
        axs[0, 1].set_xlabel('Time')
        axs[0, 1].set_ylabel('v')
        axs[0, 1].grid(True, alpha=0.3)

        axs[1, 0].plot(x, v, color='tab:green', lw=1.5)
        axs[1, 0].set_title('Phase Portrait (x-v)')
        axs[1, 0].set_xlabel('x')
        axs[1, 0].set_ylabel('v')
        axs[1, 0].grid(True, alpha=0.3)

        axs[1, 1].plot(t, te, color='tab:red', lw=1.8, label='Total E')
        if show_components.value:
            axs[1, 1].plot(t, ke, '--', color='tab:purple', alpha=0.8, label='Kinetic K')
            axs[1, 1].plot(t, pe, '--', color='tab:brown', alpha=0.8, label='Potential U')
        axs[1, 1].set_title('Energy vs Time')
        axs[1, 1].set_xlabel('Time')
        axs[1, 1].set_ylabel('Energy')
        axs[1, 1].grid(True, alpha=0.3)
        axs[1, 1].legend()

        plt.tight_layout()
        plt.show()

        display(Markdown(f'**Natural frequency:** $\\omega_0 = {omega0:.4f}$ rad/s'))
        display(Markdown(f'**Critical damping:** $c_c = 2\\sqrt{{km}} = {c_critical:.4f}$'))
        display(Markdown(f'**Regime:** {regime}'))
        display(Markdown(f'**Max |x|:** {np.max(np.abs(x)):.4f}'))
        display(Markdown(f'**Max |v|:** {np.max(np.abs(v)):.4f}'))


for w in [mass_slider, spring_slider, damp_slider, x0_slider, v0_slider, tmax_slider, dt_slider, show_components]:
    w.observe(update_plot, names='value')

update_plot()
display(out)

Output()